# Smartlead campaign and mailbox analysis

## tl;dr

- Double down first on **DealThreads staffing owners** and **Creative Alternatives law/retreat audiences**; they have the strongest positive-reply evidence.
- Do not scale Race Season or Trade Show Exhibitors unchanged: they generate replies, but few or no positive outcomes.
- **159 of 233 mailboxes have zero active campaign assignments**. The idle capacity is concentrated in DealThreads, Vantage Outbound, Calendar Group, and Restoration Homes.

## Context & Methods

This notebook records a read-only snapshot from Smartlead on August 23, 2026. Campaign comparisons use the 12 campaigns with at least one send; portfolio counts retain all 38 campaigns. Mailbox usage is based on Smartlead's shared-campaign status.

### Key Assumptions

- A positive reply is the primary outcome; opens and raw replies are supporting indicators.
- Smartlead's displayed reply-rate denominator is not re-derived because the campaign list does not expose its exact denominator.
- Recent mailbox bounce percentages are treated as directional only where Smartlead reports insufficient volume for a health grade.

## Data

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'campaign_performance.csv').exists():
    ROOT = Path('smartlead_analysis_2026-08-23')
campaigns = pd.read_csv(ROOT / 'campaign_performance.csv')
mailboxes = pd.read_csv(ROOT / 'mailbox_family_capacity.csv')
portfolio = pd.read_csv(ROOT / 'portfolio_summary.csv')
exceptions = pd.read_csv(ROOT / 'mailbox_exceptions.csv')
campaigns.shape, mailboxes.shape, portfolio.shape, exceptions.shape

### 1. Validate inputs

In [ ]:
assert len(campaigns) == 12
assert int(portfolio.set_index('metric').loc['campaigns_total', 'value']) == 38
assert mailboxes['accounts'].sum() == 233
assert mailboxes['idle_no_active'].sum() == 159
assert mailboxes['daily_capacity'].sum() == 5835
assert campaigns['sent'].sum() == 65822
assert campaigns['replies'].sum() == 277
assert campaigns['positive_replies'].sum() == 14
'All reconciliation checks passed'

## Results

### 2. Rank campaigns by positive-outcome evidence

In [ ]:
def decision(row):
    if row.positive_replies >= 3 and row.positive_share_of_replies_pct >= 3.5:
        return 'Double down'
    if row.positive_replies >= 1 and row.positive_share_of_replies_pct >= 5:
        return 'Scale carefully'
    if row.replies >= 10 and row.positive_replies == 0:
        return 'Rewrite before scaling'
    if row.positive_replies >= 1:
        return 'Optimize before scaling'
    return 'Hold / insufficient positive signal'

campaigns['decision'] = campaigns.apply(decision, axis=1)
ranked = campaigns.sort_values(['positive_replies', 'positive_share_of_replies_pct', 'replies'], ascending=False)
ranked[['campaign','status','sent','replies','positive_replies','positive_share_of_replies_pct','decision']]

### 3. Quantify idle capacity by brand family

In [ ]:
mailboxes['idle_share'] = mailboxes['idle_no_active'] / mailboxes['accounts']
mailboxes['healthy_available'] = mailboxes['idle_no_active'] - mailboxes['disconnected']
mailboxes.sort_values('idle_no_active', ascending=False)[['family','accounts','idle_no_active','idle_share','daily_capacity','disconnected','never_assigned']]

### 4. Produce the recommended first-wave allocation

In [ ]:
allocation = pd.DataFrame([
    {'family':'DealThreads','first_wave_mailboxes':25,'daily_send_ceiling':500,'motion':'Resume Staffing Owners (General); hold the other 25 for a clean adjacent-segment test'},
    {'family':'Vantage Outbound','first_wave_mailboxes':10,'daily_send_ceiling':250,'motion':'Dedicated Vantage offer pilot across healthy domains; exclude disconnected/high-bounce exceptions'},
    {'family':'Calendar Group','first_wave_mailboxes':10,'daily_send_ceiling':250,'motion':'Dedicated Calendar Group pilot after offer and prior sending issue are confirmed'},
    {'family':'Restoration Homes','first_wave_mailboxes':3,'daily_send_ceiling':60,'motion':'One mailbox per domain only after ICP and offer are confirmed'},
    {'family':'Creative Alternatives','first_wave_mailboxes':0,'daily_send_ceiling':0,'motion':'No net-new sender scale; reallocate existing active senders toward law/retreat winners'}
])
allocation

## Takeaways

1. **Resume DealThreads Staffing Owners (General) first.** It generated 5 positive replies and the strongest positive share among scaled campaigns.
2. **Concentrate Creative Alternatives on law and retreat-season audiences.** Law National produced 3 positives at scale; Law Firm Administrators is an early supporting signal.
3. **Rewrite before scaling reply-heavy/positive-light campaigns.** Race Season has 11 replies and zero positives; Trade Show Exhibitors has 50 replies but only one positive.
4. **Activate unused brands through isolated pilots.** Vantage, Calendar Group, and Restoration Homes should not borrow Creative Alternatives messaging or sender identity.